# 📊 03 — RAG Evaluation Benchmark

This notebook runs a structured benchmark of the Financial RAG Assistant using
**Claude as a judge** (`claude-haiku-4-5-20251001`) to score responses across four dimensions:

| Dimension | Definition | Pass threshold |
|-----------|-----------|----------------|
| **Grounding** | Every claim supported by retrieved passages | ≥ 0.70 |
| **Relevance** | Answer directly addresses the question | ≥ 0.70 |
| **Faithfulness** | No information beyond the context | ≥ 0.80 |
| **Completeness** | All answerable aspects are covered | ≥ 0.60 |

> **Note:** This notebook uses **simulated benchmark data** that mirrors the output
> structure of `RAGEvaluator.evaluate()` and `evaluate_batch()`. To run against
> the live pipeline, set `USE_LIVE_PIPELINE = True` and ensure API keys are set in `.env`.

**Run order:** `embeddings.py` → `02_retrieval_analysis.ipynb` → this notebook

## 0. Setup

In [ ]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from dotenv import load_dotenv

sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

# ── Config ────────────────────────────────────────────────────────────────────
USE_LIVE_PIPELINE = False   # Set True to call real pipeline + Claude judge
RANDOM_SEED       = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

COLORS    = {'tesla': '#E31937', 'apple': '#888888', 'microsoft': '#00A4EF'}
DIM_COLORS = {
    'grounding':    '#58a6ff',
    'relevance':    '#3fb950',
    'faithfulness': '#e6b450',
    'completeness': '#bc8cff',
}
THRESHOLDS = {'grounding': 0.70, 'relevance': 0.70,
               'faithfulness': 0.80, 'completeness': 0.60}
DIMS = list(THRESHOLDS.keys())

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'axes.grid':        True,
    'font.family':      'DejaVu Sans',
})

print('✅ Setup complete')

## 1. Benchmark test suite

20 representative queries covering:
- Single-company factual questions
- Cross-company comparisons
- Temporal / multi-year questions
- Edge cases (off-topic, vague)

In [ ]:
TEST_CASES = [
    # ── Tesla ─────────────────────────────────────────────────────────────────
    {"query": "What are Tesla's main risk factors?",
     "company": "tesla",   "category": "risk"},
    {"query": "How did Tesla describe its competition in the EV market?",
     "company": "tesla",   "category": "competitive"},
    {"query": "What is Tesla's strategy for Gigafactory expansion?",
     "company": "tesla",   "category": "strategy"},
    {"query": "How has Tesla addressed supply chain risks over the years?",
     "company": "tesla",   "category": "risk"},
    {"query": "What AI-related risks has Tesla disclosed?",
     "company": "tesla",   "category": "risk"},

    # ── Apple ─────────────────────────────────────────────────────────────────
    {"query": "How did Apple describe iPhone revenue trends?",
     "company": "apple",   "category": "financial"},
    {"query": "What are Apple's main geographic revenue risks?",
     "company": "apple",   "category": "risk"},
    {"query": "How does Apple describe its Services segment growth?",
     "company": "apple",   "category": "financial"},
    {"query": "What regulatory risks does Apple face in the EU?",
     "company": "apple",   "category": "risk"},
    {"query": "How has Apple's R&D spending evolved?",
     "company": "apple",   "category": "financial"},

    # ── Microsoft ─────────────────────────────────────────────────────────────
    {"query": "How has Microsoft Azure revenue grown year over year?",
     "company": "microsoft", "category": "financial"},
    {"query": "What cybersecurity risks does Microsoft disclose?",
     "company": "microsoft", "category": "risk"},
    {"query": "How does Microsoft describe its cloud strategy?",
     "company": "microsoft", "category": "strategy"},
    {"query": "What were Microsoft's key acquisitions and their risks?",
     "company": "microsoft", "category": "strategy"},
    {"query": "How has Microsoft described AI integration in its products?",
     "company": "microsoft", "category": "strategy"},

    # ── Cross-company comparisons ──────────────────────────────────────────────
    {"query": "Compare R&D spending across Tesla, Apple, and Microsoft",
     "company": "all",      "category": "comparison"},
    {"query": "How do all three companies describe competition risks?",
     "company": "all",      "category": "comparison"},
    {"query": "Which company has the highest revenue growth according to the 10-Ks?",
     "company": "all",      "category": "comparison"},

    # ── Edge cases ────────────────────────────────────────────────────────────
    {"query": "What was the stock price of Tesla last week?",
     "company": "none",     "category": "off_topic"},
    {"query": "Tell me about the CEO of Apple",
     "company": "apple",    "category": "vague"},
]

print(f'Test suite: {len(TEST_CASES)} queries')
pd.DataFrame(TEST_CASES).groupby(['company', 'category']).size().rename('count').to_frame()

## 2. Run evaluation

If `USE_LIVE_PIPELINE = False`, realistic scores are **simulated** using controlled noise
so results are reproducible without API keys.

In [ ]:
def simulate_scores(test_case: dict) -> dict:
    """
    Generate realistic simulated evaluation scores.
    Scores vary by query category to reflect expected pipeline behaviour:
      - factual / single-company queries score highest
      - comparisons are slightly harder for completeness
      - off-topic queries are blocked by guardrails (no score)
    """
    category = test_case['category']

    # Base score profiles per category
    profiles = {
        'financial':   dict(grounding=0.92, relevance=0.90, faithfulness=0.91, completeness=0.84),
        'risk':        dict(grounding=0.89, relevance=0.88, faithfulness=0.90, completeness=0.81),
        'strategy':    dict(grounding=0.85, relevance=0.86, faithfulness=0.88, completeness=0.78),
        'competitive': dict(grounding=0.83, relevance=0.87, faithfulness=0.86, completeness=0.76),
        'comparison':  dict(grounding=0.81, relevance=0.84, faithfulness=0.85, completeness=0.70),
        'vague':       dict(grounding=0.74, relevance=0.72, faithfulness=0.83, completeness=0.62),
        'off_topic':   None,   # blocked by guardrail
    }

    base = profiles.get(category)
    if base is None:
        return {**test_case, 'blocked': True, 'average': None}

    noise = {dim: np.random.normal(0, 0.04) for dim in DIMS}
    scores = {dim: float(np.clip(base[dim] + noise[dim], 0.0, 1.0)) for dim in DIMS}
    scores['average'] = round(sum(scores.values()) / 4, 3)
    scores['passed']  = all(scores[d] >= THRESHOLDS[d] for d in DIMS)
    scores['blocked'] = False

    return {**test_case, **scores}


if USE_LIVE_PIPELINE:
    from src.pipeline.pipeline import RAGPipeline
    from src.evaluation.evaluator import evaluate_batch

    pipeline  = RAGPipeline()
    live_cases = []
    for tc in TEST_CASES:
        result = pipeline.ask(tc['query'], evaluate=True)
        if result.get('blocked'):
            live_cases.append({**tc, 'blocked': True, 'average': None})
        else:
            ev = result['evaluation']
            live_cases.append({
                **tc,
                'grounding':    ev.grounding,
                'relevance':    ev.relevance,
                'faithfulness': ev.faithfulness,
                'completeness': ev.completeness,
                'average':      round(ev.average, 3),
                'passed':       ev.passed,
                'blocked':      False,
            })
    results = live_cases
else:
    results = [simulate_scores(tc) for tc in TEST_CASES]

df = pd.DataFrame(results)
df_valid = df[~df['blocked']].copy()

print(f'Total queries   : {len(df)}')
print(f'Blocked (guardrail): {df["blocked"].sum()}')
print(f'Evaluated       : {len(df_valid)}')
print(f'Pass rate       : {df_valid["passed"].mean()*100:.1f}%')
print(f'Mean avg score  : {df_valid["average"].mean():.3f}')

## 3. Results table

In [ ]:
display_cols = ['query', 'company', 'category', 'grounding', 'relevance',
                'faithfulness', 'completeness', 'average', 'passed', 'blocked']

def color_score(val):
    if pd.isna(val):
        return 'color: #8b949e'
    if val >= 0.85:
        return 'color: #3fb950; font-weight: bold'
    if val >= 0.70:
        return 'color: #e6b450'
    return 'color: #f85149; font-weight: bold'

def color_pass(val):
    if val is True:  return 'color: #3fb950'
    if val is False: return 'color: #f85149'
    return ''

styled = (
    df[display_cols]
    .style
    .applymap(color_score, subset=['grounding', 'relevance', 'faithfulness', 'completeness', 'average'])
    .applymap(color_pass, subset=['passed'])
    .format({
        'grounding': '{:.3f}', 'relevance': '{:.3f}',
        'faithfulness': '{:.3f}', 'completeness': '{:.3f}',
        'average': '{:.3f}',
    }, na_rep='—')
    .set_caption('Benchmark results — all 20 queries')
)
styled

## 4. Score distributions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle('Score distributions per dimension', fontsize=14, color='#e6edf3', y=1.02)

for ax, dim in zip(axes, DIMS):
    scores = df_valid[dim].dropna()
    ax.hist(scores, bins=8, color=DIM_COLORS[dim], alpha=0.85, edgecolor='#0d1117', linewidth=0.8)
    ax.axvline(THRESHOLDS[dim], color='#f85149', linestyle='--', linewidth=1.5,
               label=f'threshold ({THRESHOLDS[dim]})')
    ax.axvline(scores.mean(), color='#e6edf3', linestyle='-', linewidth=1.2,
               label=f'mean ({scores.mean():.2f})')
    ax.set_title(dim.capitalize(), color='#e6edf3', fontsize=11)
    ax.set_xlabel('Score', fontsize=9)
    ax.legend(fontsize=7, facecolor='#161b22', edgecolor='#30363d')
    ax.set_xlim(0.5, 1.0)

axes[0].set_ylabel('Queries', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/eval_distributions.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → docs/eval_distributions.png')

## 5. Radar chart — average scores per company

In [ ]:
companies = ['tesla', 'apple', 'microsoft']
company_means = {
    c: df_valid[df_valid['company'] == c][DIMS].mean().tolist()
    for c in companies
}

N     = len(DIMS)
theta = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
theta += theta[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True),
                        facecolor='#0d1117')
ax.set_facecolor('#161b22')

for company, means in company_means.items():
    values = means + means[:1]
    ax.plot(theta, values, color=COLORS[company], linewidth=2, label=company.capitalize())
    ax.fill(theta, values, color=COLORS[company], alpha=0.12)

# Threshold polygon
thr_vals = [THRESHOLDS[d] for d in DIMS] + [THRESHOLDS[DIMS[0]]]
ax.plot(theta, thr_vals, color='#f85149', linestyle='--', linewidth=1.2, label='Min threshold')

ax.set_xticks(theta[:-1])
ax.set_xticklabels([d.capitalize() for d in DIMS], color='#e6edf3', fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['0.6', '0.7', '0.8', '0.9', '1.0'], color='#8b949e', fontsize=8)
ax.grid(color='#30363d', linestyle='--')
ax.set_title('Radar: evaluation scores by company', color='#e6edf3',
              fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
           facecolor='#161b22', edgecolor='#30363d', fontsize=10)

plt.tight_layout()
plt.savefig('../docs/eval_radar.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → docs/eval_radar.png')

## 6. Score heatmap — query × dimension

In [ ]:
heat = df_valid.set_index('query')[DIMS].sort_values('grounding', ascending=False)
labels = [q[:55] + '…' if len(q) > 55 else q for q in heat.index]

fig, ax = plt.subplots(figsize=(10, 8), facecolor='#0d1117')
ax.set_facecolor('#161b22')

im = ax.imshow(heat.values, cmap='RdYlGn', vmin=0.55, vmax=1.0, aspect='auto')

ax.set_xticks(range(len(DIMS)))
ax.set_xticklabels([d.capitalize() for d in DIMS], fontsize=11)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)

for i in range(len(heat)):
    for j, dim in enumerate(DIMS):
        val = heat.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color='#0d1117' if val > 0.75 else '#e6edf3')

cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.ax.yaxis.set_tick_params(color='#e6edf3')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#e6edf3')

ax.set_title('Evaluation heatmap — query × dimension', color='#e6edf3',
              fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('../docs/eval_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → docs/eval_heatmap.png')

## 7. Pass rate by query category

In [ ]:
cat_stats = (
    df_valid
    .groupby('category')
    .agg(
        n        = ('query', 'count'),
        pass_rate= ('passed', 'mean'),
        avg_score= ('average', 'mean'),
    )
    .sort_values('avg_score', ascending=False)
    .reset_index()
)

cat_stats['pass_rate_pct'] = (cat_stats['pass_rate'] * 100).round(1)
cat_stats['avg_score']     = cat_stats['avg_score'].round(3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')

# Pass rate
bars1 = ax1.barh(cat_stats['category'], cat_stats['pass_rate_pct'],
                  color=['#3fb950' if p == 100 else '#e6b450' if p >= 75 else '#f85149'
                         for p in cat_stats['pass_rate_pct']],
                  edgecolor='#0d1117', linewidth=0.8)
ax1.axvline(80, color='#f85149', linestyle='--', linewidth=1.2, label='80% reference')
ax1.set_xlabel('Pass rate (%)')
ax1.set_title('Pass rate by category', color='#e6edf3', fontsize=12)
ax1.set_xlim(0, 110)
for bar, val in zip(bars1, cat_stats['pass_rate_pct']):
    ax1.text(val + 1.5, bar.get_y() + bar.get_height() / 2,
             f'{val:.0f}%', va='center', fontsize=9)
ax1.legend(fontsize=8, facecolor='#161b22', edgecolor='#30363d')

# Average score
bars2 = ax2.barh(cat_stats['category'], cat_stats['avg_score'],
                  color='#58a6ff', edgecolor='#0d1117', linewidth=0.8)
ax2.set_xlabel('Average score')
ax2.set_title('Mean score by category', color='#e6edf3', fontsize=12)
ax2.set_xlim(0.5, 1.0)
for bar, val in zip(bars2, cat_stats['avg_score']):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=9)

for ax in (ax1, ax2):
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

plt.suptitle('Query category analysis', color='#e6edf3', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../docs/eval_by_category.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → docs/eval_by_category.png')

## 8. Aggregate summary

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total queries', 'Blocked by guardrail', 'Evaluated',
        'Overall pass rate', 'Mean avg score',
        'Mean grounding', 'Mean relevance',
        'Mean faithfulness', 'Mean completeness',
        'Lowest scoring query',
    ],
    'Value': [
        len(df),
        int(df['blocked'].sum()),
        len(df_valid),
        f"{df_valid['passed'].mean()*100:.1f}%",
        f"{df_valid['average'].mean():.3f}",
        f"{df_valid['grounding'].mean():.3f}",
        f"{df_valid['relevance'].mean():.3f}",
        f"{df_valid['faithfulness'].mean():.3f}",
        f"{df_valid['completeness'].mean():.3f}",
        df_valid.loc[df_valid['average'].idxmin(), 'query'][:60] + '…',
    ]
})

display(summary.style
    .set_caption('Pipeline benchmark — summary statistics')
    .hide(axis='index'))

# Also export to JSON for CI or tracking over time
export = {
    'pass_rate':     round(df_valid['passed'].mean(), 3),
    'mean_average':  round(df_valid['average'].mean(), 3),
    **{f'mean_{d}': round(df_valid[d].mean(), 3) for d in DIMS},
    'n_evaluated':   len(df_valid),
    'n_blocked':     int(df['blocked'].sum()),
}
Path('../data').mkdir(exist_ok=True)
with open('../data/eval_summary.json', 'w') as f:
    json.dump(export, f, indent=2)

print('\nExported → data/eval_summary.json')
print(json.dumps(export, indent=2))

## ✅ Checklist before shipping

- [ ] All evaluated queries have `average ≥ 0.75`
- [ ] `faithfulness ≥ 0.80` across all categories (no hallucinations)
- [ ] Off-topic query is correctly blocked by the guardrail
- [ ] Comparison queries show ≥ 0.70 completeness (multi-source retrieval is working)
- [ ] `data/eval_summary.json` saved for tracking regressions
- [ ] Four plots saved to `docs/` and linked in README

If every box is checked → the RAG pipeline is ready for production use.